In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, ttest_ind

from CIT.evaluation.utils import load_jsonl

In [ ]:
folder_all_scores="./src/CIT/training/models/cv/scores"
all_scores_list= []
for dir in os.listdir(folder_all_scores):
    #check if it contains a directory
    if len(os.listdir(os.path.join(folder_all_scores, dir))) == 6:
        model= dir
        scores = load_jsonl(os.path.join(folder_all_scores, dir, "all_scores.jsonl"))
        all_scores_list.append({"model": model, "scores": scores})
    elif len(os.listdir(os.path.join(folder_all_scores, dir))) <5:#contains sub folders
        for subdir in os.listdir(os.path.join(folder_all_scores, dir)):
            if len(os.listdir(os.path.join(folder_all_scores, dir, subdir))) == 6:
                model= f"{dir}/{subdir}" if subdir!="ft" else dir
                scores = load_jsonl(os.path.join(folder_all_scores, dir, subdir, "all_scores.jsonl"))
                all_scores_list.append({"model": model, "scores": scores})


In [ ]:
def process_scores(scores_dict:dict):
    cleaned_scores_dict = {}


    scores=scores_dict["scores"]
    model=scores_dict["model"]
    cleaned_scores_dict["model"] = model
    for k in range(len(scores)-1):#k+1 is the fold number
        for key in scores[k].keys():
            key_fold = f"{key}_fold_{k+1}"
            cleaned_scores_dict[key_fold] = scores[k][key]

    mean_scores=scores[-1]
    for key in mean_scores.keys():
        key_mean = f"{key}_mean"
        key_std = f"{key}_std"
        cleaned_scores_dict[key_mean] = mean_scores[key]["mean"]
        cleaned_scores_dict[key_std] = mean_scores[key]["std"]
    return cleaned_scores_dict
        

In [ ]:
metric_per_model=[process_scores(scores_dict) for scores_dict in all_scores_list]
df = pd.DataFrame(metric_per_model)

In [ ]:
def compute_f1_score(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

In [ ]:
df["f1_score_mean"]= df.apply(lambda row: compute_f1_score(row["mean_precision_mean"], row["mean_recall_mean"]), axis=1)

In [ ]:
interesting_models=["base", "base_70b","ft_27.5_2epochs","ft_27.5_1epochs","ft_02.06_with_at_least_1_url_eval_all_data"]

In [ ]:
def get_one_metric_results(df:pd.DataFrame, metric:str="mean_precision",interesting_models: list=interesting_models) -> pd.DataFrame:
    """
    Get the results for a specific metric from the DataFrame.
    """
    metric_mean = f"{metric}_mean"
    metric_std = f"{metric}_std"
    good_metrics = [col for col in df.columns if metric in col]
    df_filtered= df[["model"]+good_metrics].sort_values(by=metric_mean, ascending=False)
    df_filtered = df_filtered.rename(columns={metric_mean: f"{metric} (mean)", metric_std: f"{metric} (std)"})
    df_filtered = df_filtered.reset_index(drop=True)

    df_metric = df_filtered[df_filtered["model"].isin(interesting_models)].set_index("model")
    df_metric=df_metric.T[interesting_models]
    return df_metric

In [ ]:
interesting_models=["base", "base_70b","ft_27.5_1epochs","ft_27.5_2epochs","ft_02.06_with_at_least_1_url_eval_all_data"]
map_naming_model = {
    "base": "BASE",
    "base_70b": "BASE 70B",
    "ft_27.5_1epochs": "FT1",
    "ft_27.5_2epochs": "FT2",
    "ft_02.06_with_at_least_1_url_eval_all_data": "FT2_wo0"
}


In [ ]:
df_metric=get_one_metric_results(df, metric="mean_recall",interesting_models=interesting_models)
# df_metric = df_metric[df_metric["model"].isin(interesting_models)].set_index("model")
# df_metric=df_metric.T
df_metric

In [ ]:
df_metric=df_metric.T
cols_folds_scores=[metric for metric in df_metric.columns if "fold" in metric]
df_metric["model"] = df_metric.index
df_metric_folds = df_metric[["model"] + cols_folds_scores].set_index("model")
scores_base = df_metric_folds.loc["base"].values
scores_ft = df_metric_folds.loc["ft_02.06_with_at_least_1_url_eval_all_data"].values
t_stat, p_value = ttest_ind(scores_base, scores_ft)
print(f"T-statistic: {t_stat}, P-value: {p_value}")

In [ ]:
#stacked bar plot for all models with their mean proportion_good_answers, proportion_partially_good-answers, proportion_bad_answers
def plot_stacked_bar(df: pd.DataFrame, models="all"):
    metrics= ["proportion_good_answers_mean", "proportion_partially_good_answers_mean", "proportion_bad_answers_mean"]
    renaming_metrics_map= {
        "proportion_good_answers_mean": "Good Answers",
        "proportion_partially_good_answers_mean": "Partially Good Answers",
        "proportion_bad_answers_mean": "Bad Answers"
    }
    metrics=[renaming_metrics_map[metric] for metric in metrics]
    df = df.rename(columns=renaming_metrics_map)
    df_plot = df[["model"] + metrics]
    if models != "all":
        df_plot = df_plot[df_plot["model"].isin(models)]
    df_plot = df_plot.set_index("model")
    if models!= "all":
        #reorder the models based on the list of models
        df_plot = df_plot.reindex(models)

    #rename the models for better readability
    df_plot = df_plot.rename(index=map_naming_model)
    df_plot.plot(kind='bar', stacked=True, figsize=(12, 6))
    #hline for the max of the mean proportion_good_answers+ mean proportion_partially_good_answers
    max_value = df_plot[["Good Answers", "Partially Good Answers"]].sum(axis=1).max()
    plt.axhline(y=max_value, color='r', linestyle='--', label='Best model (proportion of correct answers)')
    base_value = df_plot[["Good Answers", "Partially Good Answers"]].loc["BASE"].sum()
    plt.axhline(y=base_value, color='g', linestyle='--', label='Base nodel (proportion of correct answers)')
    #plt.title("Proportion of answers quality by model")
    plt.xlabel("", fontsize=12)
    plt.ylabel("Proportion")
    plt.xticks(rotation=45, fontsize=16)
    plt.legend(title="", bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=16)
    plt.tight_layout()
    plt.show()
plot_stacked_bar(df,models=interesting_models)
    

# correlation with previous metrics

In [ ]:
all_proportion_correct_answers = df[[metric for metric in df.columns if "proportion_correct_answers_fold" in metric]]
#flatten the values
all_proportion_correct_answers=all_proportion_correct_answers.values.flatten()
#same for  precision and recall
all_precision = df[[metric for metric in df.columns if "mean_precision_fold" in metric]]
all_precision = all_precision.values.flatten()
all_recall = df[[metric for metric in df.columns if "mean_recall_fold" in metric]]
all_recall = all_recall.values.flatten()
# Create a DataFrame for the metrics
metrics_df = pd.DataFrame({
    'Proportion Correct Answers': all_proportion_correct_answers,
    'Precision': all_precision,
    'Recall': all_recall
})
#plot histogram of the metrics on 3 subplots
fig,ax= plt.subplots(1, 3, figsize=(15, 5))
metrics_df.hist(ax=ax, bins=20, edgecolor='black', alpha=0.7)
plt.suptitle('Distribution of Metrics') 
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()



In [ ]:
corr= pearsonr(all_proportion_correct_answers, all_precision)
print(f"Pearson correlation between Proportion Correct Answers and Precision: {corr[0]}, p-value: {corr[1]:.4f}")
corr.confidence_interval(0.95)

In [ ]:
corr= pearsonr(all_proportion_correct_answers, all_recall)
print(f"Pearson correlation between Proportion Correct Answers and Recall: {corr[0]}, p-value: {corr[1]:.4f}")
corr.confidence_interval(0.95)

## PLot one metric against another

In [ ]:
df

In [ ]:
def plot_metric_against_metric(df, metric1, metric2):
    """
    Plot one metric against another.
    
    Parameters:
    df (DataFrame): DataFrame containing the metrics.
    metric1 (str): The first metric to plot.
    metric2 (str): The second metric to plot.
    """
    
    df_metric=df[[metric1, metric2, "model"]]
    df_metric = df_metric.set_index("model")
    df_metric = df_metric.rename(index=map_naming_model)
    plt.figure(figsize=(10, 6))
    plt.scatter(df_metric[metric1], df_metric[metric2], alpha=0.7)
    plt.title(f"{metric1} vs {metric2}")
    plt.xlabel(metric1)
    plt.ylabel(metric2)
    for i, txt in enumerate(df_metric.index):
        plt.annotate(txt, (df_metric[metric1].iloc[i], df_metric[metric2].iloc[i]), fontsize=9, ha='right')
  
    max_xtick = df_metric[metric1].max() + 0.05
    min_xtick = df_metric[metric1].min() - 0.05
    plt.xlim(min_xtick, max_xtick)
    max_ytick = df_metric[metric2].max() + 0.05
    min_ytick = df_metric[metric2].min() - 0.05
    plt.ylim(min_ytick, max_ytick)

    plt.tight_layout()
    plt.show()

In [ ]:
df_interesting = df[df["model"].isin(interesting_models)]

In [ ]:
plot_metric_against_metric(df_interesting, "f1_score_mean", "proportion_good_answers_mean")

# CHeck answer by answer

In [ ]:
def load_all_answers_df(answers_path,model_name="base"):
    """
    Load all answers from a given path.
    
    Parameters:
    answers_path (str): Path to the directory containing answer files.
    
    Returns:
    dict: Dictionary with filenames as keys and loaded JSON data as values.
    """
    all_answers = []
    for filename in os.listdir(answers_path):
        if filename.endswith(".jsonl"):
            file_path = os.path.join(answers_path, filename)
            fold= int(filename.split(".jsonl")[0][-1])
            answers= load_jsonl(file_path)
            for answer in answers:
                answer["fold"]= fold
                answer["f1_score"] = compute_f1_score(answer["precision"], answer["recall"])
                answer["retrieved_urls"] =len(answer["RAG_confluence_urls"])
                answer["complexity"]=len(answer["urls"])
            all_answers.extend(answers)
    answers_df = pd.DataFrame(all_answers,columns=["question_rephrased_id","quality","precision","recall","f1_score","fold","retrieved_urls","complexity","question","RAG_answer"])
    answers_df["model"] = model_name
    return answers_df

In [ ]:
answers_base="./src/CIT/training/models/cv/data/test_answers/base"
answers_base70="./src/CIT/training/models/cv/data/test_answers/base_70b"
answers_ft1="./src/CIT/training/models/cv/data/test_answers/ft_27.5_1epochs/ft"
answers_ft2="./src/CIT/training/models/cv/data/test_answers/ft_27.5_2epochs/ft"
answers_ft2_wo0="./src/CIT/training/models/cv/data/test_answers/ft_02.06_with_at_least_1_url_eval_all_data/ft"
answers_base = load_all_answers_df(answers_base, model_name="BASE")
answers_base70 = load_all_answers_df(answers_base70, model_name="BASE_70")
answers_ft1 = load_all_answers_df(answers_ft1, model_name="FT_1")
answers_ft2 = load_all_answers_df(answers_ft2, model_name="FT_2")
answers_ft2_wo0 = load_all_answers_df(answers_ft2_wo0, model_name="FT_2_wo0")
answers_df = pd.concat([answers_base, answers_base70, answers_ft1, answers_ft2, answers_ft2_wo0], ignore_index=True)

In [ ]:
answers_df.sample(2)

In [ ]:
answers_df[["quality","precision","recall","f1_score"]].corr()

In [ ]:
corr=pearsonr(answers_df["f1_score"], answers_df["quality"])
print(f"Pearson correlation between F1 Score and Quality: {corr[0]}, p-value: {corr[1]:.4f}")
corr.confidence_interval(0.95)

In [ ]:
def plot_comparison_for_metric(metric,answers_df,against_model="BASE",c_transform=False,ax=None,legend=True):
    pivot_df= answers_df.pivot_table(index="question_rephrased_id", columns="model", values=metric)
    compared_models= [col for col in pivot_df.columns if col != against_model]
    results=[]
    for model in compared_models:
        diff= pivot_df[model] - pivot_df[against_model]
        better_than_base = diff[diff > 0].count()
        worse_than_base = diff[diff < 0].count()
        if c_transform and against_model=="BASE" and (model=="FT_2_wo0" or model=="BASE_70"):
            print(f"Applying c-transform for {model} against {against_model}")
            worse_than_base-=70
            better_than_base-=40
        results.append({
            "model": model,
            "better_than": better_than_base,
            "worse_than": worse_than_base})
    results_df = pd.DataFrame(results)
    results_df
    if ax is None:
        
        results_df.plot(x="model", kind="bar", figsize=(10, 6), rot=0, 
                    title=f"Comparison of Models: Better vs Worse than {against_model} for {metric}",
                    ylabel="Number of Questions",
                    xlabel="Model",
                    legend=legend,
                    color=["green", "red"],
                    edgecolor='black',
                    width=0.8)
        plt.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
        plt.xticks(rotation=45, fontsize=12)
        plt.yticks(fontsize=12)
        plt.tight_layout()
        plt.show()
    else:
        results_df.plot(x="model", kind="bar", ax=ax, rot=0, 
                    title=f"Comparison of Models: Better vs Worse than {against_model} for {metric}",
                    ylabel="Number of Questions",
                    xlabel="Model",
                    legend=legend,
                    color=["green", "red"],
                    edgecolor='black',
                    width=0.8)
        if legend:
            ax.legend(fontsize=15, loc='upper left')
        ax.set_ylabel("Number of Questions", fontsize=16)
        ax.set_xlabel("Model", fontsize=16)
        ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, fontsize=16)
        ax.tick_params(axis='y', labelsize=12)
        ax.set_title(metric, fontsize=16)
        


In [ ]:
plot_comparison_for_metric("quality",answers_df,against_model="BASE",c_transform=False)

In [ ]:
#plot for metrics qulity precision, recall, f1 score
ids_filtered= answers_df[(answers_df["complexity"]==1)]["question_rephrased_id"].values
df= answers_df[answers_df["question_rephrased_id"].isin(ids_filtered)]
fig,ax= plt.subplots(2, 2, figsize=(15, 10),sharex=True)
plot_comparison_for_metric("quality",df,against_model="BASE",c_transform=False, ax=ax[0,0],legend=True)
plot_comparison_for_metric("precision",df,against_model="BASE",c_transform=False, ax=ax[0,1],legend=False)
plot_comparison_for_metric("recall",df,against_model="BASE",c_transform=False, ax=ax[1,0],legend=False)
plot_comparison_for_metric("f1_score",df,against_model="BASE",c_transform=False, ax=ax[1,1],legend=False)
fig.suptitle("Comparison of Models: Better vs Worse than BASE for each metric", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
#plot quality comparison for compexity from 0 to 3
ids_0= answers_df[(answers_df["complexity"]==0)]["question_rephrased_id"].values
ids_1= answers_df[(answers_df["complexity"]==1)]["question_rephrased_id"].values
ids_2= answers_df[(answers_df["complexity"]==2)]["question_rephrased_id"].values
ids_3= answers_df[(answers_df["complexity"]==3)]["question_rephrased_id"].values
df_0= answers_df[answers_df["question_rephrased_id"].isin(ids_0)]
df_1= answers_df[answers_df["question_rephrased_id"].isin(ids_1)]
df_2= answers_df[answers_df["question_rephrased_id"].isin(ids_2)]
df_3= answers_df[answers_df["question_rephrased_id"].isin(ids_3)]
fig,ax= plt.subplots(4, 1, figsize=(10, 15),sharex=True)
plot_comparison_for_metric("quality",df_0,against_model="BASE",c_transform=False, ax=ax[0],legend=True)
ax[0].set_title("Complexity 0", fontsize=16)
plot_comparison_for_metric("quality",df_1,against_model="BASE",c_transform=False, ax=ax[1],legend=False)
ax[1].set_title("Complexity 1", fontsize=16)
plot_comparison_for_metric("quality",df_2,against_model="BASE",c_transform=False, ax=ax[2],legend=False)
ax[2].set_title("Complexity 2", fontsize=16)
plot_comparison_for_metric("quality",df_3,against_model="BASE",c_transform=False, ax=ax[3],legend=False)
ax[3].set_title("Complexity 3", fontsize=16)
for complexity in range(4):
    ax[complexity].set_title(f"Complexity {complexity}: {len(np.unique(answers_df[answers_df['complexity'] == complexity]['question_rephrased_id']))} questions", fontsize=14)
#fig.suptitle("Comparison of Models: Better vs Worse than BASE for Quality for each complexity", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
def plot_all_metrics(answers_df,against_model="BASE",c_transform=False,metrics=["quality","precision","recall","f1_score"]):
    fig,ax= plt.subplots(len(metrics), 1, figsize=(10, 15),sharex=True)
    for i, metric in enumerate(metrics):
        plot_comparison_for_metric(metric,answers_df,against_model=against_model,c_transform=c_transform, ax=ax[i],legend=(i==0))
    #fig.suptitle("Comparison of Models: Better vs Worse than BASE for each metric", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
with_retrieved_urls_id= answers_df[(answers_df["retrieved_urls"]==1) & (answers_df["model"]=="FT_2_wo0")]["question_rephrased_id"].values
df_with_retrieved_urls= answers_df[answers_df["question_rephrased_id"].isin(with_retrieved_urls_id)]
plot_all_metrics(df_with_retrieved_urls,against_model="BASE",c_transform=False,metrics=["precision","quality"])



In [ ]:
plot_all_metrics(answers_df,against_model="BASE",c_transform=False,metrics=["precision","recall"])

In [ ]:
def plot_comparison_for_metric_on_each_fold(metric, model1,model_ref,answers_df,ax=None):
    """
    Plot the comparison of two models for a specific metric on each fold.
    
    Parameters:
    metric (str): The metric to compare.
    model1 (str): The first model to compare.
    model_ref (str): The second model to compare.
    answers_df (DataFrame): DataFrame containing the answers.
    """
    results = []
    for fold in list(answers_df["fold"].unique()):
        pivot_df = answers_df[answers_df["fold"] == fold].pivot_table(index="question_rephrased_id", columns="model", values=metric)
        if model1 not in pivot_df.columns or model_ref not in pivot_df.columns:
            print(f"Model {model1} or {model_ref} not found in fold {fold}. Skipping.")
            continue
        diff = pivot_df[model1] - pivot_df[model_ref]
        better_than_ref = diff[diff > 0].count()
        worse_than_ref = diff[diff < 0].count()
        
        result = {"fold": fold,
                   "better_than": better_than_ref,
                   "worse_than": worse_than_ref}
        results.append(result)

    results_df = pd.DataFrame(results)
    results_df.set_index("fold", inplace=True)
    results_df.sort_index(inplace=True)
    if ax is None:
        results_df.plot(kind="bar", figsize=(10, 6), rot=0,
                        title=f"Comparison of {model1} vs {model_ref} for {metric} on Each Fold",
                        ylabel="Number of Questions",
                        xlabel="Fold",
                        legend=True,
                        color=["green", "red"],
                        edgecolor='black',
                        width=0.8)
        plt.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
        plt.xticks(rotation=45, fontsize=12)
        plt.yticks(fontsize=12)
        plt.tight_layout()
        plt.show()
    else:
        results_df.plot(kind="bar", ax=ax, rot=0,
                        title=f"Comparison of {model1} vs {model_ref} for {metric} on Each Fold",
                        ylabel="Number of Questions",
                        xlabel="Fold",
                        legend=True,
                        color=["green", "red"],
                        edgecolor='black',
                        width=0.8)
        ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, fontsize=12)
        ax.tick_params(axis='y', labelsize=12)


In [ ]:
answers_df.model.unique()

In [ ]:
plot_comparison_for_metric_on_each_fold("quality","FT_2_wo0","BASE",answers_df)

In [ ]:
#check question in 5th fold where FT_2_wo0 performs worse than BASE:
fold_5 = answers_df[answers_df.fold == 4]
#the metric is in the column quality and the model names in the column model
pivot = fold_5.pivot(index="question_rephrased_id", columns="model", values="quality")
diff = pivot["FT_2_wo0"] - pivot["BASE"]
worse_than_base = diff[diff < 0]
indx_worse = worse_than_base.index



In [ ]:
random_idx = np.random.randint(0, len(indx_worse) - 1)
random_idx = indx_worse[random_idx]
sample_base=answers_base[answers_base["question_rephrased_id"] == random_idx]
sample_ft=answers_ft2_wo0[answers_ft2_wo0["question_rephrased_id"] == random_idx]

#print question, then answer for both base en ft models
print("Question:", sample_base["question"].values[0])
print("Base Answer:", sample_base["RAG_answer"].values[0])
print("---"*20)
print("FT question:", sample_ft["question"].values[0])
print("FT Answer:", sample_ft["RAG_answer"].values[0])

# metrics depending on threshold, for topk=6

In [ ]:
exp_folder="./src/CIT/training/models/cv/scores/change_threshold"
#load from subfolders
all_scores_list= []
for dir in os.listdir(exp_folder):
    #check if it contains a directory
    if len(os.listdir(os.path.join(exp_folder, dir))) == 6:
        threshold= int(dir.split("threshold")[-1])/100
        scores = load_jsonl(os.path.join(exp_folder, dir, "all_scores.jsonl"))
        scores=scores[-1]  # take the last element which contains the mean scores
        all_scores_list.append({"threshold": threshold, "scores": scores})

def process_scores_threshold(scores_dict:dict):
    cleaned_scores_dict = {}
    scores=scores_dict["scores"]
    threshold=scores_dict["threshold"]
    cleaned_scores_dict["threshold"] = threshold
    for key in scores.keys():
        cleaned_scores_dict[key] = scores[key]["mean"]
    return cleaned_scores_dict

metric_per_model_threshold=[process_scores_threshold(scores_dict) for scores_dict in all_scores_list]
df_threshold = pd.DataFrame(metric_per_model_threshold)
df_threshold.sort_values(by="threshold", inplace=True)
df_threshold = df_threshold.set_index("threshold")
df_threshold["f1_score"] = df_threshold.apply(lambda row: compute_f1_score(row["mean_precision"], row["mean_recall"]), axis=1)

In [ ]:
df_threshold.plot(y=["proportion_correct_answers","proportion_good_answers","mean_precision", "mean_recall", "f1_score"], figsize=(10, 6), marker='o')
plt.title("Metrics vs Threshold")
plt.xlabel("Threshold")
plt.ylabel("Metric Value")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

## grouping by complexity

In [ ]:
int(0)/100

In [ ]:
# look at all answers, group results by complexity
folder_answers="./src/CIT/training/models/cv/data/test_answers/change_threshold_ft"
all_answers = []
for dir in os.listdir(folder_answers):
    if os.path.isdir(os.path.join(folder_answers, dir)):
        threshold= int(dir.split("threshold")[-1])/100
        for file in os.listdir(os.path.join(folder_answers, dir)):
            if file.endswith(".jsonl"):
                file_path = os.path.join(folder_answers, dir, file)
                answers = load_jsonl(file_path)
                for answer in answers:
                    answer["threshold"] = threshold
                    answer["complexity"] = len(answer["urls"])
                    answer["correct_answer"] = answer["quality"] >0
                    answer["good_answer"] = answer["quality"] == 2
                all_answers.extend(answers)

answers_df = pd.DataFrame(all_answers, columns=["precision", "recall", "complexity", "quality", "threshold","correct_answer", "good_answer","question_rephrased_id"])
answers_df["f1_score"] = answers_df.apply(lambda row: compute_f1_score(row["precision"], row["recall"]), axis=1)

In [ ]:
answers_df[(answers_df['complexity'] == 1)]['question_rephrased_id'].nunique()

In [ ]:
# Group by complexity and threshold, then calculate mean values
grouped_df = answers_df.drop("question_rephrased_id",axis=1).groupby(["complexity", "threshold"]).mean().reset_index()
grouped_df = grouped_df.pivot(index="complexity", columns="threshold", values=["precision", "recall", "f1_score", "correct_answer", "good_answer"])
#plot the values on 4 plots for the 4 complexities
fig, axs = plt.subplots(4,1, figsize=(12, 16), sharex=True)
for i, complexity in enumerate(grouped_df.index):
    axs[i].plot(grouped_df.columns.levels[1], grouped_df.loc[complexity, ("precision", slice(None))], marker='o', label='Precision')
    axs[i].plot(grouped_df.columns.levels[1], grouped_df.loc[complexity, ("recall", slice(None))], marker='o', label='Recall')
    axs[i].plot(grouped_df.columns.levels[1], grouped_df.loc[complexity, ("f1_score", slice(None))], marker='o', label='F1 Score')
    axs[i].plot(grouped_df.columns.levels[1], grouped_df.loc[complexity, ("correct_answer", slice(None))], marker='o', label='Correct Answers')
    axs[i].plot(grouped_df.columns.levels[1], grouped_df.loc[complexity, ("good_answer", slice(None))], marker='o', label='Good Answers')
    axs[i].set_title(f"Complexity {complexity} - {answers_df[(answers_df['complexity'] == complexity)]['question_rephrased_id'].nunique()} questions")
    axs[i].set_xlabel("Threshold")
    axs[i].set_ylabel("Metric Value")
    axs[i].grid(True)
    axs[i].legend()
    axs[i].set_xticks(grouped_df.columns.levels[1])
    axs[i].set_xticklabels(grouped_df.columns.levels[1], rotation=45)
plt.suptitle("Metrics vs Threshold for Different Complexities")
plt.tight_layout()
plt.show()